In [2]:
import pandas as pd
import numpy as np



In [3]:
casu_data=pd.read_excel('/home/johnkel/Desktop/Casaurina trial/causaurina_2006_data.xlsx')
casu_data.head()

,Block,T/NO,DBH (cm),HT (m),Spacing
0,I,1,D,D,1x1
1,I,2,6.5,7.5,1x1
2,I,3,D,D,1x1
3,I,4,D,D,1x1
4,I,5,D,D,1x1


In [4]:
# Rename columns to be easier to work with in code (no spaces/parentheses)
casu_data = casu_data.rename(columns={
    "T/NO": "TreeNo",
    "DBH (cm)": "DBH",
    "HT (m)": "Height"
})

In [5]:
# A tree is dead if either DBH or Height is recorded as "D"
casu_data["Survival"] = casu_data["DBH"].ne("D").astype(int)#this creates anew column called "survival" that acts as a bool to indicate is a tree is dead or alive
# Survival = 1 (alive), 0 (dead)

In [6]:
casu_data.head()

,Block,TreeNo,DBH,Height,Spacing,Survival
0,I,1,D,D,1x1,0
1,I,2,6.5,7.5,1x1,1
2,I,3,D,D,1x1,0
3,I,4,D,D,1x1,0
4,I,5,D,D,1x1,0


In [7]:
#Now convert DBH and Height to proper numeric columns
# errors="coerce" turns "D" (and anything else non-numeric) into NaN
casu_data["DBH"] = pd.to_numeric(casu_data["DBH"], errors="coerce")
casu_data["Height"] = pd.to_numeric(casu_data["Height"], errors="coerce")
#this converts values in both two columns into numeric so as to not be interpreted  as strings


In [8]:
 # Make Block and Spacing categorical
casu_data["Block"] = casu_data["Block"].astype("category")
casu_data["Spacing"] = casu_data["Spacing"].astype("category")
#makes them categorical variables so that they can be use in statistical modles

In [9]:
print(casu_data.dtypes)

Block       category
TreeNo         int64
DBH          float64
Height       float64
Spacing     category
Survival       int64
dtype: object


In [10]:
print(casu_data["Survival"].value_counts())
#Checking the number of alive and dead trees in the dataset. where 1=alive and 0=dead

Survival
1    738
0    452
Name: count, dtype: int64


In [11]:
print(casu_data.groupby(["Spacing", "Block"])["Survival"].mean())  # survival rate per group

Spacing  Block
1x1      I        0.638723
         II       0.604839
2.5x2.5  I        0.580000
         II       0.645161
Name: Survival, dtype: float64


Explanatory data Analysis

In [12]:
# Keeping only alive trees for growth stats (dead trees have no DBH/Height to measure)
alive = casu_data[casu_data["Survival"] == 1]


In [13]:
# ── Survival rate by group ───────────────────────────────────────
survival_by_group = casu_data.groupby(["Spacing", "Block"], observed=True)["Survival"].agg(
    survival_rate="mean", n_trees="count")

print("Survival rate by Spacing x Block:")
print(survival_by_group)
print()

Survival rate by Spacing x Block:
               survival_rate  n_trees
Spacing Block                        
1x1     I           0.638723      501
        II          0.604839      496
2.5x2.5 I           0.580000      100
        II          0.645161       93



In [14]:
# ── DBH summary stats by group (alive trees only) ───────────────
print("DBH (cm) summary by Spacing x Block:")
print(alive.groupby(["Spacing", "Block"], observed=True)["DBH"].describe())
print()

DBH (cm) summary by Spacing x Block:
               count      mean       std  min    25%   50%    75%  max
Spacing Block                                                         
1x1     I      320.0  4.116250  1.332468  1.1  3.000  4.20  5.000  8.5
        II     300.0  4.481667  1.211413  1.5  3.500  4.50  5.500  7.6
2.5x2.5 I       58.0  4.977586  1.626323  2.0  3.500  5.25  6.375  8.6
        II      60.0  4.916667  1.492940  2.0  3.775  5.00  6.275  7.5



In [ ]:
# Check for mismatches: DBH says one thing, Height says another
mismatch_data = casu_data[(casu_data["DBH"] == "D") != (casu_data["Height"] == "D")]
print(f"Rows where DBH and Height disagree on dead status: {len(mismatch_data)}")
print(mismatch_data[["TreeNo", "Block", "Spacing", "DBH", "Height"]])

Rows where DBH and Height disagree on dead status: 0
Empty DataFrame
Columns: [TreeNo, Block, Spacing, DBH, Height]
Index: []


In [16]:
print("Height (m) summary by Spacing x Block:")
print(alive.groupby(["Spacing", "Block"], observed=True)["Height"].describe())

Height (m) summary by Spacing x Block:
               count      mean       std   min     25%   50%   75%  max
Spacing Block                                                          
1x1     I      320.0  4.782531  1.525747  1.50  3.5000  4.75  6.00  8.0
        II     298.0  5.635235  1.724355  1.40  4.5000  6.00  7.00  8.5
2.5x2.5 I       58.0  5.153448  1.851478  1.75  3.5625  5.25  6.75  8.0
        II      60.0  5.163333  1.895599  1.75  3.7500  5.00  7.00  8.0
